## Data preprocessing

In [38]:
import json
import numpy as np
import nltk
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.models import KeyedVectors
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import random

nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    """Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
    tokens = word_tokenize(text.lower())
    filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
    return ' '.join(filtered_tokens)

def text2seq(text, tokenizer, max_length):
    sequences = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequences, maxlen=max_length, padding='post')
    return padded[0] if padded.shape[0] > 0 else np.zeros(max_length)
    
# Load JSON data
def load_data(filepath):
    with open(filepath, 'r') as file:
        data = json.load(file)
    return data


[nltk_data] Downloading package punkt to /Users/clarec/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [16]:

claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')

claims = [info['claim_text'] for info in claims_data.values()]
evidences = [evidence for evidence in evidence_data.values()]


In [ ]:
# evidence_map = {eid: preprocess_text(text) for eid, text in evidence_data.items()}

# def save_to_json(filepath, data):
#     """ Save a dictionary to a JSON file. """
#     with open(filepath, 'w', encoding='utf-8') as f:
#         json.dump(data, f, ensure_ascii=False, indent=4)

# # Example usage
# save_to_json('data/curated/preprocessed_evidence_map.json', evidence_map)

In [35]:

evidence_map = load_data('data/curated/preprocessed_evidence_map.json')


In [39]:
data_for_dataframe = []
evidence_keys = list(evidence_map.keys())  # List of all evidence IDs

for claim_id, claim_details in claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    claim_evidences = set(claim_details['evidences'])  # Convert to set for faster checks

    # Add positive examples
    for eid in claim_evidences:
        evidence_text = evidence_map.get(eid, "NULL")  
        if evidence_text != "NULL":
            data_for_dataframe.append({
                'claim': claim_text,
                'evidence': evidence_text,
                'label': 1  # Label as relevant
            })

    # Add negative examples
    num_neg_samples = min(len(claim_evidences), len(evidence_keys) - len(claim_evidences))  # Limit the number of negative samples
    negative_samples = random.sample([k for k in evidence_keys if k not in claim_evidences], num_neg_samples)
    for eid in negative_samples:
        evidence_text = evidence_map[eid]
        data_for_dataframe.append({
            'claim': claim_text,
            'evidence': evidence_text,
            'label': 0  # Label as not relevant
        })


In [40]:
df = pd.DataFrame(data_for_dataframe)

# Initialize and fit the tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['claim'].tolist() + df['evidence'].tolist())

# Maximum length of text sequences
max_length = max(df['claim'].str.split().str.len().max(), df['evidence'].str.split().str.len().max())

# Apply text to sequence conversion
df['claim_seq'] = df['claim'].apply(lambda x: text2seq(x, tokenizer, max_length))
df['evidence_seq'] = df['evidence'].apply(lambda x: text2seq(x, tokenizer, max_length))

claims_input = np.stack(df['claim_seq'].values)
evidences_input = np.stack(df['evidence_seq'].values)
labels = df['label'].values


## Creating the Embedding Matrix

In [41]:
# Load pre-trained word2vec
word_vectors = KeyedVectors.load('word2vec.wordvectors', mmap='r')
vocab_size = len(tokenizer.word_index) + 1

# Create an embedding matrix
embedding_dim = 50  # dimension of word2vec vectors
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in word_vectors:
        embedding_vector = word_vectors[word]
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector


## Building the LSTM Model

We will build a simple unidirectional LSTM model to compare claim and evidence embeddings.

In [43]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Dropout, concatenate
from tensorflow.keras import regularizers

# Define the model
def create_model():
        claims_input = Input(shape=(max_length,), dtype='int32')
        evidences_input = Input(shape=(max_length,), dtype='int32')

        # Shared Embedding layer
        embedding_layer = Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=False)

        # LSTM layers
        claims_embeddings = embedding_layer(claims_input)
        evidences_embeddings = embedding_layer(evidences_input)

        claims_lstm = LSTM(64)(claims_embeddings)
        evidences_lstm = LSTM(16)(evidences_embeddings)

        # Concatenate and output
        concatenated = concatenate([claims_lstm, evidences_lstm])
        concatenated = Dense(64, kernel_regularizer=regularizers.l2(0.001), activation='relu')(concatenated)
        concatenated = Dropout(0.5)(concatenated)
        output = Dense(1, activation='sigmoid')(concatenated)  

    
        model = Model(inputs=[claims_input, evidences_input], outputs=output)
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

        return model

model = create_model()
print(model.summary())


Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_11 (InputLayer)       [(None, 177)]                0         []                            
                                                                                                  
 input_12 (InputLayer)       [(None, 177)]                0         []                            
                                                                                                  
 embedding_3 (Embedding)     (None, 177, 50)              721100    ['input_11[0][0]',            
                                                                     'input_12[0][0]']            
                                                                                                  
 lstm_6 (LSTM)               (None, 64)                   29440     ['embedding_3[0][0]']   

###  Training the Model

In [44]:

# Train the model
model.fit([claims_input, evidences_input], labels, epochs=10, batch_size=32, validation_split=0.1)


Epoch 1/10
232/232 [==============================] - 11s 45ms/step - loss: 0.7120 - accuracy: 0.5082 - val_loss: 0.6945 - val_accuracy: 0.5006
Epoch 2/10
232/232 [==============================] - 8s 35ms/step - loss: 0.6937 - accuracy: 0.4941 - val_loss: 0.6933 - val_accuracy: 0.5006
Epoch 3/10
232/232 [==============================] - 9s 38ms/step - loss: 0.6934 - accuracy: 0.4980 - val_loss: 0.6932 - val_accuracy: 0.4994
Epoch 4/10
232/232 [==============================] - 9s 39ms/step - loss: 0.6933 - accuracy: 0.4968 - val_loss: 0.6932 - val_accuracy: 0.4994
Epoch 5/10
232/232 [==============================] - 9s 39ms/step - loss: 0.6933 - accuracy: 0.4843 - val_loss: 0.6932 - val_accuracy: 0.4994
Epoch 6/10
232/232 [==============================] - 11s 49ms/step - loss: 0.6932 - accuracy: 0.4952 - val_loss: 0.6932 - val_accuracy: 0.5006
Epoch 7/10
232/232 [==============================] - 8s 35ms/step - loss: 0.6932 - accuracy: 0.4949 - val_loss: 0.6932 - val_accuracy: 0.49

## Test the model

In [54]:
claim_text = "The Earth’s climate sensitivity is so low that a doubling of atmospheric CO2 will result in a surface temperature change on the order of 1°C or less."
claim_seq = text2seq(preprocess_text(claim_text), tokenizer, max_length)
claim_seq = np.expand_dims(claim_seq, 0)
print("Claims input shape:", claims_input.shape)  # Should be (number of evidences, max_length)

evidences_list = ["In his first paper on the matter, he estimated that global temperature would rise by around 5 to 6 °C (9.0 to 10.8 °F) if the quantity of CO 2 was doubled", "The 1990 IPCC First Assessment Report estimated that equilibrium climate sensitivity to a dou- bling of CO 2 lay between 1.5 and 4.5 °C (2.7 and 8.1 °F), with a \"best guess in the light of current knowledge\" of 2.5 °C (4.5 °F).", "John Bennet Lawes, English entrepreneur and agricultural scientist"] 
# evidence_seqs = np.array([text2seq(preprocess_text(ev), tokenizer, max_length) for ev in evidences_list])

# claims_input = np.tile(claim_seq, (len(evidence_seqs), 1))

# Iterate over each evidence and predict
predictions = []
for evidence_text in evidences_list:
    evidence_seq = text2seq(preprocess_text(evidence_text), tokenizer, max_length)
    evidence_seq = np.expand_dims(evidence_seq, 0)  # Ensure it's 2D
    
    print("Evidences input shape:", evidence_seq.shape)  # Should also be (number of evidences, max_length)

    # Predict the relation between the claim and this piece of evidence
    prediction = model.predict([claim_seq, evidence_seq])
    predictions.append((evidence_text, prediction[0]))

# Display predictions
for evidence, pred in predictions:
    print(f"Evidence: {evidence}\nPrediction: {pred}\n")

Claims input shape: (3, 177)
Evidences input shape: (1, 177)
1/1 [==============================] - 0s 88ms/step
Evidences input shape: (1, 177)
1/1 [==============================] - 0s 13ms/step
Evidences input shape: (1, 177)
1/1 [==============================] - 0s 13ms/step
Evidence: In his first paper on the matter, he estimated that global temperature would rise by around 5 to 6 °C (9.0 to 10.8 °F) if the quantity of CO 2 was doubled
Prediction: [0.50049275]

Evidence: The 1990 IPCC First Assessment Report estimated that equilibrium climate sensitivity to a dou- bling of CO 2 lay between 1.5 and 4.5 °C (2.7 and 8.1 °F), with a "best guess in the light of current knowledge" of 2.5 °C (4.5 °F).
Prediction: [0.50049275]

Evidence: John Bennet Lawes, English entrepreneur and agricultural scientist
Prediction: [0.50049275]



In [56]:
# threshold = 0.5  # This threshold can be adjusted based on your validation set performance or specific needs
# related_evidences = [(evidences_list[i], predictions[i][0]) for i in range(len(predictions)) if predictions[i] > threshold]

# print("Related Evidences:")
# for ev, score in related_evidences:
#     print(f"Evidence: {ev} \n\tScore: {score}")


In [27]:
def batch_predict(claims, evidences, model, batch_size=32):
    results = []
    for i in range(0, len(evidences), batch_size):
        batch_evidences = evidences[i:i+batch_size]
        batch_claims = np.tile(claims, (len(batch_evidences), 1))
        batch_preds = model.predict([batch_claims, batch_evidences])
        results.extend(batch_preds)
    return results

# Use batch predict function
predictions = batch_predict(claim_seq, evidence_seqs, model)
related_evidences = [(evidences_list[i], predictions[i][0]) for i in range(len(predictions)) if predictions[i] > threshold]
related_evidences

1/1 [==============================] - 0s 18ms/step


[('In his first paper on the matter, he estimated that global temperature would rise by around 5 to 6 °C (9.0 to 10.8 °F) if the quantity of CO 2 was doubled',
  0.5127229),
 ('The 1990 IPCC First Assessment Report estimated that equilibrium climate sensitivity to a dou- bling of CO 2 lay between 1.5 and 4.5 °C (2.7 and 8.1 °F), with a "best guess in the light of current knowledge" of 2.5 °C (4.5 °F).',
  0.5127229),
 ('John Bennet Lawes, English entrepreneur and agricultural scientist',
  0.5127229)]